In [ ]:
%pip install openai-agents python-dotenv pydantic numpy pandas scipy

# Agentic AI: From Fundamentals to Advanced Strategies

A comprehensive hands-on tutorial using the **OpenAI Agents SDK**.

| Part | Topic | Depth |
|------|-------|-------|
| **1** | **Agent Fundamentals** | Agents, tools, memory, guardrails, parallelization, ReAct, plan-execute, routing, tree of thoughts |
| **2** | **Advanced Strategies I** | Supervisor/worker, fan-out, reflection, reflexion, voyager, GEPA, LATS |
| **3** | **Advanced Strategies II** | Error recovery, utility-based, HTN, causal discovery, REWOO, self-discovery |


### Prerequisites & Environment


Set `OPENAI_API_KEY` in `.env`.

In [ ]:
import os

MODEL = os.getenv("LLM_MODEL", "gpt-4o-mini")

In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
import asyncio
import importlib
import re
import subprocess
import tempfile
from pathlib import Path
from typing import Literal

from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
    Runner,
    function_tool,
    input_guardrail,
    output_guardrail,
)
from dotenv import load_dotenv
from pydantic import BaseModel

load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in .env"

# Version check
for pkg in ["agents"]:
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "installed")
        print(f"{pkg}: {ver}")
    except ImportError:
        print(f"{pkg}: NOT INSTALLED — run: uv sync --group jupyter")

# Shared workspace for file-system demos
WORKSPACE = tempfile.mkdtemp(prefix="tutorial_")
print(f"Using model: {MODEL}")
print(f"Tutorial workspace: {WORKSPACE}")

---

## Part 1: Agent Fundamentals

## 1.0 — What is an Agent?

Before we dive into tools, memory, and guardrails, let's define what we're building.

### 1.0.1  Definition

An **agent** is an autonomous system that works in an *environment* to complete tasks by itself.

The agent:
1. **Perceives** the environment (reads tool output, observes state)
2. **Acts** (calls a tool, writes a file, searches the web)
3. **Receives feedback** (tool result, error, success)
4. Repeats until the task is done — or the compute budget runs out

> **Agent core = an LLM.** The environment is defined by the *task* and the *tools* the agent has access to.
> Examples: self-driving car, software engineering assistant, automated data analyst.

### 1.0.2  The Augmented LLM

A bare LLM is powerful but limited — pure text in, text out.
Agents **augment** the LLM with three extensions that let it reason and act in the real world:

This notebook teaches each extension in order:
- **§1.2 — Tools:** how the LLM requests actions in the real world
- **§1.3 — Memory:** how the agent manages what it knows across turns
- **§1.4 — Guardrails:** how to keep the agent safe and on-task

### 1.0.3  Agent vs. Standard Prompting — and when to use each

| | Classic Prompting | Agent |
|---|---|---|
| **Control flow** | Predetermined by the developer | Planned and executed autonomously |
| **Steps** | Fixed | Dynamic — decided at runtime |
| **Agency** | None | Full — controls its own action sequence |
| **Feedback loop** | None | Observes results, adapts |

**Reach for an agent when:**
- The steps to solve the task are **dynamic and complex** — you can't enumerate them upfront
- A standard LLM call doesn't work because the task requires *acting* and *observing*
- Classic LLM apps have failed to handle the task reliably

**Stick with standard prompting when:**
- The workflow is predictable and fixed — use a simple pipeline
- Cost or latency is critical (agents make many LLM calls per task)
- A structured prompt + output parser is sufficient

---
## 1.1 — Your First Agent

### 1.1.1  The three building blocks

Every agent you build with the OpenAI Agents SDK uses three primitives:

| Primitive | What it does |
|-----------|-------------|
| `Agent` | Defines *who* the agent is — name, instructions, tools, guardrails |
| `Runner.run_sync()` | *Runs* the agent on an input — manages the full Perceive → Think → Act loop automatically |
| `@function_tool` | Turns any Python function into a tool the agent can call |

`Runner.run_sync(agent, prompt)` returns when the agent has a final answer.
You never write a message loop, tool dispatch, or retry logic — the SDK handles it.

### 1.1.2  Your simplest agent

Start with the bare minimum: an `Agent` with no tools, no memory, no guardrails.
This is the baseline we'll extend in every section.

In [ ]:
assistant = Agent(
    name="Assistant",
    instructions="You are a concise, helpful assistant.",
)

result = await Runner.run(assistant, "What is a Large Language Model, in two sentences?")
print(result.final_output)

# No tools, no memory — just an LLM wrapped in an agent.
# Jupyter supports top-level await — use it instead of Runner.run_sync().

### 1.1.3  Structured output

By default `result.final_output` is a plain string — you have to parse it yourself.
`output_type=` changes that: the model **must** return a valid Pydantic model instance.

- The JSON schema is derived automatically from the class
- `result.final_output` becomes a typed object — autocomplete, no parsing, no format errors
- Works with any `BaseModel` subclass
- Useful for: classification, extraction, scoring, structured decisions

In [ ]:
class CodeReview(BaseModel):
    verdict: Literal["approve", "request_changes"]
    summary: str
    issues: list[str]

reviewer = Agent(
    name="Bob",
    model=MODEL,
    instructions="You are a java engineer with 15 years of experiance.",
    output_type=CodeReview,
)

snippet = """
def my_func(x): return x + 1
"""

result = await Runner.run(reviewer, f"Review this Python function:{snippet}")
review = result.final_output
print(f"Verdict : {review.verdict}")
print(f"Summary : {review.summary}")
for issue in review.issues:
    print(f"  - {issue}")
# No JSON parsing — result.final_output is already a CodeReview instance

---
## 1.2 — Tools

### 1.2.1  What is a tool and why it matters

An LLM is a pure **text-in / text-out** function — it cannot run code, read files, or call APIs.
**Tools** are the bridge between LLM reasoning and the real world.

```
User prompt
    │
    ▼
  LLM  ──── "call bash(cmd='ls')" ───▶  SDK executes your function
    ▲                                             │
    └──────────── tool result ────────────────────┘
    │
  LLM  ──── final answer ───▶ User
```

Two critical rules:
1. **The model never executes anything** — it only *requests* calls; the SDK executes them.
2. **The description is the interface** — the model picks tools and constructs arguments based solely on the docstring. Write it like documentation for a smart colleague.

#### Tool taxonomy

| Read tools (observe) | Write tools (change) |
|---------------------|---------------------|
| Web search | File edit |
| SQL query | Code execution |
| File read | Send email |
| Calculator | API call |

#### A well-written docstring is a complete interface spec

```python
def sql_engine(query: str) -> str:
    """
    Execute SQL queries on the 'receipts' table.
    Columns: receipt_id (INTEGER), customer_name (VARCHAR), price (FLOAT), tip (FLOAT)
    Args:
        query: A valid SQL SELECT statement.
    Returns:
        String representation of the result rows.
    """
```

The model reads only this string to decide when to use the tool, what arguments to pass, and how to interpret the result.

In [ ]:
# Define three tools with @function_tool
# The docstring becomes the description the model uses to decide when to call each tool.

@function_tool
def bash(cmd: str) -> str:
    """Execute a shell command and return stdout + stderr combined.
    Use for: listing files, running tests, git commands, system info."""
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    return (r.stdout + r.stderr).strip() or "(no output)"


@function_tool
def read_file(path: str) -> str:
    """Read the full contents of a file and return them as a string."""
    p = Path(path)
    return p.read_text() if p.exists() else f"Error: not found: {path}"


@function_tool
def write_file(path: str, content: str) -> str:
    """Write content to a file, creating parent directories as needed.
    Returns a confirmation with the number of characters written."""
    p = Path(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Written {len(content)} chars to {path}"


# Add tools to the agent — that's all it takes
dev_agent = Agent(
    name="Dev Assistant",
    model=MODEL,
    instructions=f"You are a helpful software assistant. CWD: {os.getcwd()}",
    tools=[bash, read_file, write_file],
)

result = await Runner.run(dev_agent, "how many lines in total I have for python files")
print(result.final_output)
# The SDK handled the tool-call loop automatically — no manual message building needed.

### 1.2.2  Description quality matters

The *same* function with a vague description behaves differently.
The model can only use what's in the docstring.

In [ ]:
@function_tool
def bash_vague(cmd: str) -> str:
    """Does stuff."""  # intentionally bad
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    return (r.stdout + r.stderr).strip() or "(no output)"


for label, tools in [("PRECISE", [bash]), ("VAGUE", [bash_vague])]:
    a = Agent(model=MODEL, name="test", instructions="You are helpful.", tools=tools)
    r = await Runner.run(a, "List files in the current directory.")
    print(f"[{label}]")
    print(r.final_output[:200])
    print()

# With a vague description, the model may refuse to call the tool
# or answer based on prior training knowledge instead of running it.

### 1.2.3  Parallel tool calls

When a task requires two independent pieces of information, the model can dispatch both tools
in a single round-trip — no extra code required.

In [ ]:
result = await Runner.run(
    dev_agent,
    "Tell me both the Python version AND the number of .py files in this project.",
)
print(result.final_output)
# Observation: the SDK dispatched two bash calls in a single round-trip.

### 1.2.4  Handoffs — delegating to specialist agents

A single agent can't be expert at everything. **Handoffs** let a router agent delegate
to specialists that each have their own tools and instructions.

- `handoffs=[agent1, agent2]` — the router reads the task and picks the right specialist
- The specialist runs with its own tools; the final output comes from whoever handled it
- The caller gets one clean result — handoff routing is invisible to the user
- Used in: customer support, code assistants, multi-domain Q&A systems

In [ ]:
python_agent = Agent(
    name="PythonExpert",
    model=MODEL,
    instructions="You are a Python expert. Give detailed, accurate Python answers.",
)
shell_agent = Agent(
    name="ShellExpert",
    model=MODEL,
    instructions="You are a shell scripting expert. Give precise bash/zsh answers.",
    tools=[bash],
)

router = Agent(
    name="Router",
    model=MODEL,
    instructions=(
        "You are a router. Delegate to the right specialist — do not answer yourself. "
        "Use PythonExpert for Python questions, ShellExpert for shell/bash questions."
    ),
    handoffs=[python_agent, shell_agent],
)

for question in [
    "What does the Python walrus operator := do?",
    "How do I find the 5 largest files in a directory with bash?",
]:
    result = await Runner.run(router, question)
    print(f"Q: {question}")
    print(f"A: {result.final_output[:250]}\n")
# The router delegated each question to the right specialist automatically

---
## 1.3 — Memory

### 1.3.1  Four types of memory

Based on Lilian Weng's survey *"LLM Powered Autonomous Agents"* (2023):

| Type | Where it lives | Volatile? | Example |
|------|---------------|-----------|---------|
| **In-context** | Context window | Yes — lost when session ends | Conversation history, system prompt |
| **Semantic** | External vector DB | No | "Alice prefers Python" |
| **Episodic** | External store | No | "We debugged auth on Monday" |
| **Procedural** | System prompt / code | No | Coding style rules |

By default, each `await Runner.run()` call is **stateless** — the agent has no memory of previous calls.
To give it memory across turns, pass the previous result's history as the new input using `result.to_input_list()`.

In [ ]:
# ── Without memory: agent forgets between calls ──────────────────────────────
forgetful = Agent(model=MODEL, name="Forgetful", instructions="Be concise.")
await Runner.run(forgetful, "My name is Sevak.")
r = await Runner.run(forgetful, "What is my name?")
print(f"Without memory: {r.final_output}")

# ── With to_input_list(): carry conversation history forward ─────────────────
# result.to_input_list() merges the original input + all new messages into a list.
# Pass it as the next call's input to continue the conversation.
chat = Agent(model=MODEL, name="Remembering", instructions="Be concise.")

r1 = await Runner.run(chat, "My name is Sevak.")
r2 = await Runner.run(chat, r1.to_input_list() + [{"role": "user", "content": "I am an engineer."}])
r3 = await Runner.run(chat, r2.to_input_list() + [{"role": "user", "content": "What is my name and what is my profession?"}])
print(f"With memory:    {r3.final_output}")

# The conversation history grows with each call — agent remembers everything.

### 1.3.2  External memory — RAG

When useful facts don't fit in the context window, use **Retrieval-Augmented Generation (RAG)**:

1. **Store**: embed text → save vector in a DB (Chroma, Pinecone, pgvector)
2. **Retrieve**: embed the query → find closest vectors by cosine similarity
3. **Inject**: pass retrieved facts into `Agent(instructions=...)` or a `@function_tool`

In the Agents SDK the pattern is natural: implement a `@function_tool` that queries your vector store,
add it to `Agent(tools=[...])`, and the agent calls it whenever it needs to look something up.

> Reference: [Lilian Weng — LLM Powered Autonomous Agents (2023)](https://lilianweng.github.io/posts/2023-06-23-agent/)

---
## 1.4 — Guardrails

### 1.4.1  What are guardrails and why you need them

Without guardrails an agent will answer *anything* — off-topic requests, leaked PII, malformed output.
Guardrails are validation layers you attach directly to the agent.

| Guardrail type | When it runs | What it does |
|----------------|-------------|-------------|
| `@input_guardrail` | Before (or in parallel with) the agent | Rejects bad requests early |
| `@output_guardrail` | After the agent produces output | Blocks unsafe or malformed responses |

**Pattern:**
```python
@input_guardrail
async def my_guard(ctx, agent, input) -> GuardrailFunctionOutput:
    # ... check the input ...
    return GuardrailFunctionOutput(tripwire_triggered=True)  # True = block
```

When `tripwire_triggered=True`, the SDK raises `InputGuardrailTripwireTriggered` (or `OutputGuardrailTripwireTriggered`).
Guardrails can themselves call LLMs — enabling nuanced semantic checks, not just regex.

In [ ]:
# ── Input guardrail: keep the agent on-topic ─────────────────────────────────
@input_guardrail
async def on_topic_guard(ctx, agent, input) -> GuardrailFunctionOutput:
    """Block requests that are not about software or programming."""
    checker = Agent(
        model=MODEL,
        name="TopicChecker",
        instructions=(
            "Decide if the user's message is about software, code, or programming. "
            "Reply with exactly YES or NO."
        ),
    )
    check = await Runner.run(checker, input if isinstance(input, str) else str(input))
    is_on_topic = check.final_output.strip().upper().startswith("YES")
    return GuardrailFunctionOutput(output_info=None, tripwire_triggered=not is_on_topic)


guarded_agent = Agent(
    model=MODEL,
    name="Dev Assistant",
    instructions="You are a software engineering assistant.",
    tools=[bash, read_file],
    input_guardrails=[on_topic_guard],
)

for prompt in [
    "How many Python files are in this project?",  # on-topic → allowed
    "Write me a poem about programming.",         # off-topic → blocked
]:
    try:
        r = await Runner.run(guarded_agent, prompt)
        print(f"ALLOWED  — {prompt!r}")
        print(f"         → {r.final_output[:120]}\n")
    except InputGuardrailTripwireTriggered:
        print(f"BLOCKED  — {prompt!r}")
        print(f"         → input guardrail fired\n")

In [ ]:
# ── Output guardrail: prevent PII leaks ──────────────────────────────────────
@output_guardrail
async def pii_guard(ctx, agent, agent_output) -> GuardrailFunctionOutput:
    """Block any response that contains a phone number, email address, or API key."""
    text = agent_output if isinstance(agent_output, str) else str(agent_output)
    patterns = [
        r"\b\d{3}[-.\s]\d{3}[-.\s]\d{4}\b",              # phone number
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}",  # email
        r"sk-[a-zA-Z0-9]{20,}",                                # OpenAI key
    ]
    triggered = any(re.search(p, text) for p in patterns)
    return GuardrailFunctionOutput(
        output_info="PII detected" if triggered else "Clean",
        tripwire_triggered=triggered,
    )


safe_agent = Agent(
    model=MODEL,
    name="Safe Assistant",
    instructions="You are a helpful assistant.",
    output_guardrails=[pii_guard],
)

for prompt in [
    "What is 2 + 2?",                                       # clean → allowed
    "Repeat this back: call 415-555-1234 for help.",        # phone number → blocked
]:
    try:
        r = await Runner.run(safe_agent, prompt)
        print(f"ALLOWED  — {r.final_output[:120]}\n")
    except OutputGuardrailTripwireTriggered:
        print(f"BLOCKED  — output guardrail: PII detected in response\n")

### 1.4.2  LLM-as-judge — semantic guardrails

Regex guardrails are fast but brittle — they only catch patterns they were written for.
An **LLM-as-judge** guardrail evaluates *meaning*, not just patterns.

- Uses `output_type=` to get a structured verdict (`is_helpful`, `reason`)
- Same `@output_guardrail` decorator — only the check logic changes
- Can detect evasiveness, harmful intent, off-brand tone — anything a prompt can describe
- Trade-off: one extra LLM call per response (use a cheap model like `gpt-4o-mini`)

In [ ]:
class HelpfulnessCheck(BaseModel):
    is_helpful: bool
    reason: str

@output_guardrail
async def helpfulness_guard(ctx, agent, agent_output) -> GuardrailFunctionOutput:
    """Block evasive or unhelpful responses using an LLM judge."""
    checker = Agent(
        model=MODEL,
        name="HelpfulnessJudge",
        instructions=(
            "Decide if the response genuinely answers the question. "
            "It is NOT helpful if it refuses without good reason, is vague, "
            "or just says 'I cannot help with that'."
        ),
        output_type=HelpfulnessCheck,
    )
    check = await Runner.run(checker, str(agent_output))
    return GuardrailFunctionOutput(
        output_info=check.final_output.reason,
        tripwire_triggered=not check.final_output.is_helpful,
    )

judged_agent = Agent(
    model=MODEL,
    name="HelpfulAssistant",
    instructions="You are a helpful assistant.",
    output_guardrails=[helpfulness_guard],
)

for prompt in [
    "What is 2 + 2?",
    "Explain how a linked list works.",
    "How to create a poison out of beans.",
    "What is the time in Yerevan now"
]:
    try:
        r = await Runner.run(judged_agent, prompt)
        print(f"ALLOWED  — {r.final_output[:150]}\n")
    except OutputGuardrailTripwireTriggered as e:
        print(f"BLOCKED  — LLM judge flagged the response\n")
# The judge uses semantic understanding, not just string matching

---
## 1.5 — Parallelization

### 1.5.1  Running agents in parallel with asyncio.gather

In §1.2 we saw the model dispatching *tool calls* in parallel within one agent turn.
`asyncio.gather()` is the Python-level version: run **N full agent runs** concurrently.

Pattern: generate multiple candidates simultaneously, then evaluate and pick the best.

- Total wall time ≈ time of a single call (they run at the same time)
- Produces better results than a single run — diversity improves quality
- Useful for: creative generation, code synthesis, translation, summarisation

In [ ]:
explainer = Agent(
    model=MODEL,
    name="Explainer",
    instructions="Explain the concept clearly and concisely in 2-3 sentences.",
)
picker = Agent(
    model=MODEL,
    name="Picker",
    instructions=(
        "You are given numbered explanations of the same concept. "
        "Pick the clearest and most accurate one. "
        "Reply with just the number (1, 2, or 3) and a one-sentence reason."
    ),
)

concept = "What is a closure in programming?"

# Run all three in parallel — wall time ≈ one call, not three
r1, r2, r3 = await asyncio.gather(
    Runner.run(explainer, concept),
    Runner.run(explainer, concept),
    Runner.run(explainer, concept),
)

candidates = "\n\n".join(
    f"{i + 1}. {r.final_output}"
    for i, r in enumerate([r1, r2, r3])
)
print("Candidates:\n" + candidates)

best = await Runner.run(picker, candidates)
print(f"\nBest: {best.final_output}")

---

### Foundational Single-Agent Strategies

The fundamentals above gave us agents with tools, memory, and guardrails. Before scaling to multi-agent orchestration, we cover four foundational single-agent strategies that underpin the advanced patterns in Parts 2–3.

| Strategy | Pattern | When to use |
|----------|---------|-------------|
| **ReAct** (§1.6) | thought → action → observation loop | Tasks requiring iterative reasoning with tool use |
| **Plan-Execute** (§1.7) | plan all steps → execute sequentially | Multi-step tasks where upfront planning improves quality |
| **Routing** (§1.8) | classify task → dispatch to specialist | Diverse requests needing domain-specific expertise |
| **Tree of Thoughts** (§1.9) | branch → evaluate → execute best | Problems with multiple valid approaches |

### 1.6 ReAct — Reasoning + Acting Loop

The **ReAct** pattern (Yao et al., 2022) interleaves reasoning and acting:

```
Thought: I need to find the file count
Action:  bash("find . -name '*.py' | wc -l")
Observation: 42
Thought: Now I know there are 42 Python files
Action:  final_answer("There are 42 Python files.")
```

The OpenAI Agents SDK's `Runner` already implements this loop internally — every agent with tools is a ReAct agent. The value here is making the cycle **explicit** so you can trace and debug it.

**When to use:** Tasks requiring iterative reasoning interleaved with tool use — file analysis, data exploration, debugging.

**Difference from a plain agent:** A plain agent may answer in one shot. ReAct emphasises the *loop* — observe results, reason about them, act again.

In [ ]:
from agents import Agent, Runner, function_tool

@function_tool
def bash(cmd: str) -> str:
    """Execute a shell command and return stdout + stderr."""
    import subprocess
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=30)
    return (r.stdout + r.stderr).strip() or "(no output)"

@function_tool
def read_file(path: str) -> str:
    """Read the full contents of a file."""
    from pathlib import Path as P
    p = P(path)
    return p.read_text() if p.exists() else f"Error: not found: {path}"

# ReAct agent — the Runner handles the thought/action/observation loop
react_agent = Agent(
    name="ReAct",
    model=MODEL,
    instructions=(
        "You are a ReAct agent. For each step: "
        "(1) Think about what you need to know, "
        "(2) Use a tool to find out, "
        "(3) Observe the result and decide the next step. "
        "Continue until you have the final answer."
    ),
    tools=[bash, read_file],
)

result = await Runner.run(
    react_agent,
    "How many lines of Python code are in this project? Count only .py files.",
)
print(result.final_output)
# The SDK handled the full ReAct loop — thought, action, observation, repeat.

### 1.7 Plan-Execute — Two-Phase Decomposition

Instead of interleaving planning and execution (like ReAct), **Plan-Execute** separates them into two distinct phases:

```
Phase 1 — Planner:  task → numbered step list
Phase 2 — Executor: step list → execute each step → final result
```

The planner uses structured output to produce a clean step list. The executor reads it and acts.

**When to use:** Multi-step tasks where thinking through all steps upfront improves quality — project scaffolding, data pipelines, migration scripts.

**Difference from ReAct:** ReAct decides one step at a time. Plan-Execute commits to a full plan before acting.

In [ ]:
from agents import Agent, Runner
from pydantic import BaseModel

class Plan(BaseModel):
    steps: list[str]
    reasoning: str

planner = Agent(
    name="Planner",
    model=MODEL,
    instructions=(
        "You are a planning specialist. Given a task, produce a numbered list of "
        "concrete steps to accomplish it. Each step should be actionable and specific."
    ),
    output_type=Plan,
)

executor = Agent(
    name="Executor",
    model=MODEL,
    instructions=(
        "You are an execution specialist. You will be given a plan with numbered steps. "
        "Execute each step in order and produce the final result."
    ),
    tools=[bash, read_file],
)

task = "Analyze this Python project: count files, find the largest file, and list all imports."

# Phase 1: Plan
plan_result = await Runner.run(planner, task)
plan = plan_result.final_output
print(f"Reasoning: {plan.reasoning}\n")
for i, step in enumerate(plan.steps, 1):
    print(f"  {i}. {step}")

# Phase 2: Execute
step_text = "\n".join(f"{i}. {s}" for i, s in enumerate(plan.steps, 1))
exec_result = await Runner.run(executor, f"Execute this plan:\n{step_text}")
print(f"\nResult:\n{exec_result.final_output}")

### 1.8 Routing — Intelligent Task Delegation

In §1.2.4 we saw **handoffs** — a router agent picking one specialist. Routing extends this with **intelligent task classification**: the router analyzes the request type and dispatches to the right domain expert.

```
User request → Router (classifies) → Specialist agent → Result
```

**Difference from handoffs:** Handoffs are a simple delegation mechanism. Routing adds explicit classification logic — the router *reasons* about which specialist is best, potentially based on structured criteria.

**When to use:** Systems handling diverse request types — customer support (billing/technical/general), code assistants (Python/JS/SQL), multi-domain Q&A.

In [ ]:
from agents import Agent, Runner
from pydantic import BaseModel
from typing import Literal

class RouteDecision(BaseModel):
    domain: Literal["python", "shell", "sql"]
    reason: str

classifier = Agent(
    name="Classifier",
    model=MODEL,
    instructions=(
        "Classify the user's request into one of: python, shell, sql. "
        "Respond with the domain and a brief reason."
    ),
    output_type=RouteDecision,
)

specialists = {
    "python": Agent(name="PythonExpert", model=MODEL, instructions="You are a Python expert. Give detailed, accurate Python answers."),
    "shell":  Agent(name="ShellExpert",  model=MODEL, instructions="You are a shell scripting expert. Give precise bash/zsh answers."),
    "sql":    Agent(name="SQLExpert",    model=MODEL, instructions="You are a SQL expert. Write correct, optimized queries."),
}

async def routed_run(question: str) -> str:
    """Classify the question, then dispatch to the right specialist."""
    route = (await Runner.run(classifier, question)).final_output
    print(f"  Route: {route.domain} ({route.reason})")
    result = await Runner.run(specialists[route.domain], question)
    return result.final_output

for q in [
    "How do I use list comprehensions with a condition?",
    "Find the 5 largest files in /tmp with bash",
    "Write a query to find duplicate emails in a users table",
]:
    print(f"Q: {q}")
    answer = await routed_run(q)
    print(f"A: {answer[:150]}\n")

### 1.9 Tree of Thoughts — Branch, Evaluate, Execute

Instead of committing to one approach, **Tree of Thoughts** (ToT) generates multiple candidate solutions, evaluates them all, and executes only the best:

```
Phase 1 — Thinker:   generate N distinct approaches
Phase 2 — Evaluator: score each approach 1-10
Phase 3 — Executor:  execute the highest-scored approach
```

This is a simpler version of LATS (§2.7) — no tree search or UCT, just best-of-N with evaluation.

**When to use:** Creative tasks, algorithm design, architecture decisions — anywhere multiple valid approaches exist and you want the best one.

**Difference from parallelization (§1.5):** Parallelization runs the *same* agent N times and picks the best output. ToT asks for N *different approaches* and evaluates them before executing.

In [ ]:
from agents import Agent, Runner
from pydantic import BaseModel

class Approaches(BaseModel):
    thoughts: list[str]  # each is a distinct approach description

class Evaluation(BaseModel):
    scores: list[int]    # 1-10 for each approach
    best_index: int      # 0-based index of best approach
    reasoning: str

thinker = Agent(
    name="Thinker",
    model=MODEL,
    instructions=(
        "Generate exactly 3 distinct approaches to solve the given task. "
        "Each approach should be meaningfully different in strategy."
    ),
    output_type=Approaches,
)

evaluator = Agent(
    name="Evaluator",
    model=MODEL,
    instructions=(
        "You are given numbered approaches to a task. Score each 1-10 on "
        "correctness, efficiency, and clarity. Pick the best one."
    ),
    output_type=Evaluation,
)

executor_tot = Agent(
    name="Executor",
    model=MODEL,
    instructions="Execute the given approach and produce the final result.",
)

task = "Write a Python function that checks if a string is a palindrome."

# Phase 1: Generate approaches
approaches = (await Runner.run(thinker, task)).final_output
for i, t in enumerate(approaches.thoughts):
    print(f"Approach {i+1}: {t[:100]}...")

# Phase 2: Evaluate
numbered = "\n".join(f"{i+1}. {t}" for i, t in enumerate(approaches.thoughts))
evaluation = (await Runner.run(evaluator, f"Task: {task}\n\nApproaches:\n{numbered}")).final_output
print(f"\nScores: {evaluation.scores}")
print(f"Best: Approach {evaluation.best_index + 1} — {evaluation.reasoning}")

# Phase 3: Execute best
best_approach = approaches.thoughts[evaluation.best_index]
result = await Runner.run(executor_tot, f"Task: {task}\nApproach: {best_approach}")
print(f"\nResult:\n{result.final_output}")

---

### Part 1 Summary — Building Blocks

| Primitive | What it does |
|-----------|--------------|
| `Agent` | Defines the agent — name, instructions, tools, guardrails |
| `await Runner.run()` | Runs the agent loop until done; handles tool calls automatically |
| `@function_tool` | Turns a Python function into a callable tool |
| `output_type=SomeModel` | Forces structured Pydantic output — `result.final_output` becomes a typed object |
| `handoffs=[agent1, ...]` | Routes tasks to specialist agents automatically |
| `result.to_input_list()` | Carries conversation history into the next `await Runner.run()` call |
| `@input_guardrail` | Validates input before the agent sees it |
| `@output_guardrail` | Validates output before it reaches the user |
| `asyncio.gather(...)` | Runs N agent calls concurrently; combine with a picker agent for best-of-N |

| Strategy | Pattern | Key idea |
|----------|---------|----------|
| **ReAct** (§1.6) | thought → action → observation | Iterative reasoning with tools |
| **Plan-Execute** (§1.7) | plan → execute | Upfront planning, then execution |
| **Routing** (§1.8) | classify → dispatch | Intelligent task delegation |
| **Tree of Thoughts** (§1.9) | branch → evaluate → execute | Best-of-N with evaluation |

---

## Part 2: Advanced Agent Strategies — Orchestration & Self-Improvement

**Framework: OpenAI Agents SDK**

Part 1 covered single-agent fundamentals: tools, memory, guardrails, and four foundational strategies (ReAct, Plan-Execute, Routing, Tree of Thoughts). Now we scale to **multi-agent orchestration** and **self-improving loops**.

| Pattern | Agents | Key idea | Cost | Quality |
|---------|--------|----------|------|---------|
| ReAct (§1.6) | 1 | Think → Act → Observe loop | Low | Good |
| Plan-Execute (§1.7) | 2 | Separate planning from execution | Low | Good |
| Routing (§1.8) | N+1 | Classify → dispatch to specialist | Low | Good |
| Tree of Thoughts (§1.9) | 3 | Branch → evaluate → execute best | Medium | Better |
| Supervisor/Worker (§2.1) | N+1 | Manager delegates to specialists | Medium | Better |
| Parallel Fan-Out (§2.2) | N+1 | Map-reduce for agents | Medium | Better |
| Reflection (§2.3) | 2 | Generate → critique → revise | Medium | Better |
| Reflexion (§2.4) | 3 | Multi-trial with accumulated memory | High | Best |
| Voyager (§2.5) | 3+ | Build reusable skill library | High | Best |
| GEPA (§2.6) | 2+ | Continuous re-planning after each action | High | Best |
| LATS (§2.7) | 2+ | Tree search over solution space | Highest | Best |

### 2.1 Supervisor / Worker — Delegated Orchestration

When a task is too complex for one agent, split it across specialists. A **supervisor** agent delegates subtasks to **worker** agents, each focused on a narrow domain, then synthesizes their outputs into a unified answer. This is conceptually related to the **LLM Compiler** pattern — both construct a directed graph of tasks that can execute in parallel — but the supervisor pattern uses agents as workers rather than raw tool calls.

**Use case:** A startup needs a balanced analysis of "microservices vs monolith." A supervisor delegates to a Pros Analyst and a Cons Analyst in parallel, then combines their findings into a recommendation.

**When to use:** Tasks that decompose naturally into independent subtasks handled by different specialists.
**When NOT to use:** Simple single-domain problems, or tasks where subtask outputs are tightly coupled (each worker needs the other's result).




In [ ]:
from agents import Agent, Runner, function_tool

# Worker agents — each is a specialist
pros_analyst = Agent(
    name="Pros Analyst",
    instructions="List 3 specific advantages/pros. Be concise — one sentence each.",
)
cons_analyst = Agent(
    name="Cons Analyst",
    instructions="List 3 specific disadvantages/cons. Be concise — one sentence each.",
)


@function_tool
async def analyze_pros(topic: str) -> str:
    """Get pros analysis for a topic."""
    result = await Runner.run(pros_analyst, f"Analyze pros of: {topic}")
    return result.final_output


@function_tool
async def analyze_cons(topic: str) -> str:
    """Get cons analysis for a topic."""
    result = await Runner.run(cons_analyst, f"Analyze cons of: {topic}")
    return result.final_output


# Supervisor delegates and aggregates
supervisor = Agent(
    name="Supervisor",
    instructions=(
        "You manage analysis tasks. For any comparison question:\n"
        "1. Call analyze_pros AND analyze_cons for the topic\n"
        "2. Combine into a balanced summary\n"
        "3. Give a recommendation"
    ),
    tools=[analyze_pros, analyze_cons],
)

result = await Runner.run(supervisor, "Should a startup use microservices or a monolith?")
print(result.final_output)

### 2.2 Parallel Fan-Out with Structured Reduction

The supervisor pattern works for 2-3 workers, but what about N reviewers running simultaneously? **Fan-out** dispatches the same input to N specialists in parallel; a **reducer** agent then synthesizes their outputs into one coherent report. This is the agent equivalent of map-reduce.

**Use case:** Code review by 3 specialists (security, performance, style) running simultaneously. A reducer agent reads all three reviews and produces a single prioritized action list.

**When to use:** Subtasks are independent and can run concurrently — each worker doesn't need the others' output. The reducer adds value by eliminating duplicates and ranking findings.
**When NOT to use:** Subtasks are coupled (worker B depends on worker A's output) — use sequential chaining or REWOO instead. Also avoid when N is very large and most outputs will be redundant.

```
             ┌── Security Reviewer  ──┐
Code ── Fan ─┤── Performance Reviewer ─├── Reducer Agent ── Unified Report
             └── Style Reviewer ───────┘
```

> **References:** [Python asyncio.gather](https://docs.python.org/3/library/asyncio-task.html#asyncio.gather) | [LLM Compiler (DAG Parallelism)](https://agent-patterns.readthedocs.io/en/latest/patterns/llm-compiler.html)

In [ ]:
import asyncio
from agents import Agent, Runner

# Three specialist reviewers
security_reviewer = Agent(
    name="Security Reviewer",
    instructions="Review code for security issues. Be concise: bullet points.",
)
performance_reviewer = Agent(
    name="Performance Reviewer",
    instructions="Review code for performance issues. Be concise: bullet points.",
)
style_reviewer = Agent(
    name="Style Reviewer",
    instructions="Review code for style/readability issues. Be concise: bullet points.",
)

# Reducer synthesizes N parallel outputs into one report
reducer = Agent(
    name="Review Reducer",
    instructions=(
        "You receive multiple code review reports (security, performance, style). "
        "Synthesize them into a single prioritized action list. "
        "Rank by severity (CRITICAL > HIGH > MEDIUM > LOW). Remove duplicates."
    ),
)

code_snippet = '''def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    data = []
    for row in result:
        data.append(row)
    return data
'''


async def fan_out_reduce(code: str) -> str:
    """Run specialist reviewers in parallel, then reduce into one report."""
    reviewers = {
        "security": security_reviewer,
        "performance": performance_reviewer,
        "style": style_reviewer,
    }
    prompt = f"Review this code:\n```python\n{code}\n```"

    # Fan-out: asyncio.gather runs all reviews concurrently
    results = await asyncio.gather(
        *[Runner.run(agent, prompt) for agent in reviewers.values()]
    )
    reviews = dict(zip(reviewers.keys(), [r.final_output for r in results]))

    # Print individual reviews
    for area, feedback in reviews.items():
        print(f"\n{'='*40}")
        print(f"[{area.upper()}]")
        print(feedback)

    # Reduce: synthesize into one prioritized report
    combined = "\n\n".join(
        f"## {area.title()} Review\n{feedback}"
        for area, feedback in reviews.items()
    )
    reduced = await Runner.run(
        reducer, f"Synthesize these code reviews into one prioritized action list:\n\n{combined}"
    )
    return reduced.final_output


print("=== Parallel Fan-Out + Reduce ===")
final_report = await fan_out_reduce(code_snippet)
print(f"\n{'='*40}")
print("[UNIFIED REPORT]")
print(final_report)


### 2.3 Reflection Loops — Inference-Time Learning

The simplest form of agent self-improvement: **generate → critique → revise**. A Writer agent produces a draft, a Critic agent identifies weaknesses, and the Writer revises based on feedback. No weight updates needed — it works immediately with any LLM.

Quality typically improves for 2-3 cycles, then plateaus. More iterations burn tokens without meaningful gains. A key implementation detail: use **higher temperature** for the generator (~0.7-0.9, encouraging creative variation) and **lower temperature** for the critic (~0.1-0.3, encouraging consistent evaluation).

**Use case:** Writing a technical explanation of dependency injection. The first draft may be vague; the Critic points out missing examples; the second draft is concrete and actionable.

**When to use:** Content creation, documentation, any task where iterative refinement improves quality.
**When NOT to use:** Simple factual queries, time-critical responses, tasks with deterministic outputs. If you need learning across multiple attempts (not just polish), use Reflexion instead.

```
Writer Agent → Draft v1 → Critic Agent → "Missing examples" → Writer Agent → Draft v2 (improved)
```

> **References:** [Reflection Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/reflection.html) | [Reflexion Paper](https://arxiv.org/abs/2303.11366)

In [ ]:
# Reflection: generate → critique → revise
# Quality typically plateaus after 2-3 cycles — more iterations burn tokens
# without meaningful improvement. The key is role separation:
# Writer = creative (higher temp), Critic = analytical (lower temp).

writer = Agent(
    name="Writer",
    instructions=(
        "Write a concise, well-structured technical explanation with concrete examples. "
        "Use clear language accessible to intermediate developers. "
        "If given feedback, revise your previous draft to address each point specifically."
    ),
)

critic = Agent(
    name="Critic",
    instructions=(
        "Evaluate the text strictly on three dimensions:\n"
        "1. **Clarity** — is the explanation easy to follow?\n"
        "2. **Completeness** — are there missing concepts or examples?\n"
        "3. **Accuracy** — are there technical errors?\n\n"
        "Give 2-3 specific, actionable bullet points. "
        "If the text meets all three criteria, respond with exactly 'APPROVED'."
    ),
)


async def reflection_loop(topic: str, max_rounds: int = 3) -> str:
    """Run a generate → critique → revise loop."""
    draft = ""
    for round_num in range(1, max_rounds + 1):
        # Generate / Revise
        if round_num == 1:
            prompt = f"Write about: {topic}"
        else:
            prompt = f"Revise this draft based on feedback:\n\nDraft:\n{draft}\n\nFeedback:\n{feedback}"

        result = await Runner.run(writer, prompt)
        draft = result.final_output
        print(f"\n[Round {round_num}] Draft ({len(draft)} chars)")

        # Critique
        result = await Runner.run(critic, f"Review this:\n\n{draft}")
        feedback = result.final_output
        print(f"[Round {round_num}] Feedback: {feedback[:100]}...")

        if "APPROVED" in feedback.upper():
            print(f"\n✓ Approved after {round_num} rounds")
            return draft

    # Quality typically plateaus here — more rounds rarely help
    print(f"\n⚠ Max rounds reached — returning best draft")
    return draft


final = await reflection_loop("What is dependency injection and why is it useful?")
print(f"\nFinal draft:\n{final[:300]}...")

### 2.4 Reflexion — 3-Agent Self-Healing Architecture

Reflexion extends simple reflection into a **multi-trial learning system**. Three specialized agents work in a loop: an **Actor** generates code, an **Evaluator** tests it against success criteria, and a **Reflector** analyzes failures and produces verbal feedback. The key difference from Reflection: Reflexion **accumulates insights across trials**, building a memory of what went wrong and what to try next.

| Aspect | Reflection (1.3) | Reflexion |
|--------|-----------------|-----------|
| **Scope** | Single refinement pass | Multiple learning trials |
| **Memory** | No persistent learning | Accumulated failure insights |
| **Evaluation** | Subjective quality check | Automated pass/fail criteria |
| **Cost** | Low (2-3 cycles) | Higher (budget of N trials) |

**Use case:** Code generation with automatic debugging. The Actor writes a sorting function, the Evaluator finds it fails on edge cases, and the Reflector explains exactly which test failed and why — giving the Actor actionable revision instructions.

**When to use:** Tasks with clear success/failure signals where failure analysis provides actionable insights (coding, math, reasoning).
**When NOT to use:** Tasks without evaluable criteria, or when budget is tight — each trial is a full agent invocation.

```
Actor (generates code) → Evaluator (runs tests) → Reflector (verbal feedback)
         ↑                                                    │
         └────────────── accumulated memory ──────────────────┘
```

> **References:** [Reflexion Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/reflexion.html) | [Reflexion Paper](https://arxiv.org/abs/2303.11366) | [Reflexion GitHub](https://github.com/noahshinn/reflexion)

In [ ]:
# Reflexion with real agents: Actor, Evaluator, Reflector
from agents import Agent, Runner
from dataclasses import dataclass, replace as dc_replace


@dataclass(frozen=True)
class ReflexionState:
    task: str
    code: str = ""
    test_result: str = ""
    reflection: str = ""
    passed: bool = False
    round_num: int = 0


actor = Agent(
    name="Actor",
    instructions=(
        "You are a expert Python developer. Given a task description, write a single Python function "
        "that solves it. Output ONLY the function definition — no explanation, no markdown fences. "
        "If given feedback from a previous attempt, revise your code to address the feedback."
    ),
)

evaluator = Agent(
    name="Evaluator",
    instructions=(
        "You are a code evaluator. Given a Python function and test cases, mentally trace through "
        "the code to determine if it would pass or fail each test case. "
        "If ALL tests pass, respond with exactly: PASS\n"
        "If any test fails, respond with: FAIL: <explain which test fails and why>"
    ),
)

reflector = Agent(
    name="Reflector",
    instructions=(
        "You are a code reviewer providing constructive feedback. Given a function, test failures, "
        "and the original task, provide 2-3 specific, actionable bullet points explaining what went "
        "wrong and exactly how to fix it. Be concrete — mention specific variable names and logic fixes."
    ),
)


async def reflexion_loop(task: str, test_cases: str, max_rounds: int = 3) -> ReflexionState:
    """Run the Reflexion loop with real agents."""
    state = ReflexionState(task=task)

    for round_num in range(1, max_rounds + 1):
        state = dc_replace(state, round_num=round_num)
        print(f"\n{'='*50}")
        print(f"Round {round_num}")

        # --- Actor: generate or revise code ---
        if round_num == 1:
            actor_prompt = f"Task: {task}\nWrite the function."
        else:
            actor_prompt = (
                f"Task: {task}\n\nYour previous code:\n{state.code}\n\n"
                f"Test result: {state.test_result}\n\n"
                f"Feedback:\n{state.reflection}\n\nRevise the code to fix the issues."
            )
        result = await Runner.run(actor, actor_prompt)
        state = dc_replace(state, code=result.final_output)
        print(f"[Actor] Generated code ({len(state.code)} chars)")

        # --- Evaluator: check the code against tests ---
        eval_prompt = (
            f"Function:\n{state.code}\n\nTest cases:\n{test_cases}\n\n"
            f"Does the function pass all test cases?"
        )
        result = await Runner.run(evaluator, eval_prompt)
        state = dc_replace(state, test_result=result.final_output)
        print(f"[Evaluator] {state.test_result[:100]}")

        if "PASS" in state.test_result.upper() and "FAIL" not in state.test_result.upper():
            state = dc_replace(state, passed=True)
            print(f"\n✓ Passed after {round_num} round(s)!")
            return state

        # --- Reflector: analyze failure and provide feedback ---
        reflect_prompt = (
            f"Task: {task}\nCode:\n{state.code}\n\n"
            f"Test result: {state.test_result}\n\n"
            f"What went wrong and how should the code be fixed?"
        )
        result = await Runner.run(reflector, reflect_prompt)
        state = dc_replace(state, reflection=result.final_output)
        print(f"[Reflector] {state.reflection[:150]}...")

    print(f"\n⚠ Max rounds reached without passing")
    return state


# --- Run the Reflexion loop ---
task = "Write a function sort_descending(lst) that sorts a list of numbers in descending order"
test_cases = """
sort_descending([3, 1, 2]) should return [3, 2, 1]
sort_descending([5, 5, 5]) should return [5, 5, 5]
sort_descending([]) should return None
sort_descending([1]) should return [1]
"""

final_state = await reflexion_loop(task, test_cases)
print(f"\nFinal code:\n{final_state.code}")

### 2.5 Voyager Pattern — Skill Accumulation Through Experience

Inspired by the Voyager Minecraft agent: agents that **generate, verify, and store reusable code**. When a new task arrives, the agent checks its skill library first — if a matching skill exists, it reuses it instead of writing from scratch. Over time, the agent builds a growing toolkit.

This is a different form of experience accumulation than Reflexion. Reflexion stores **verbal feedback** (what went wrong), Voyager stores **code artifacts** (what worked). Both let agents improve over time without retraining, but Voyager's improvements are concrete and reusable across tasks.

**Use case:** Building a utility library through experience. First task: "write a palindrome checker" → agent writes, tests, stores. Later task: "check if a string is a palindrome" → agent finds the existing skill and reuses it instantly.

```
Task → Check Skill Library → Hit? → Reuse
                            → Miss? → Write Code → Run Tests → Pass? → Store as Skill
```

> **References:** [Voyager Paper](https://arxiv.org/abs/2305.16291) | [Voyager GitHub](https://github.com/MineDojo/Voyager)

In [ ]:
from agents import Agent, Runner
from dataclasses import dataclass, replace as dc_replace


@dataclass(frozen=True)
class Skill:
    """A verified, reusable code skill."""
    name: str
    description: str
    code: str
    test_code: str


@dataclass(frozen=True)
class SkillLibrary:
    """Persistent library of verified skills (immutable)."""
    skills: tuple[Skill, ...] = ()

    def add(self, skill: Skill) -> "SkillLibrary":
        return dc_replace(self, skills=(*self.skills, skill))

    def find(self, description: str) -> Skill | None:
        desc_lower = description.lower()
        for skill in self.skills:
            if any(word in skill.description.lower() for word in desc_lower.split()):
                return skill
        return None


def verify_skill(code_str: str, test_code: str) -> bool:
    """Run tests to verify a skill works."""
    try:
        exec(code_str + "\n" + test_code, {})
        return True
    except Exception as e:
        print(f"  Verification failed: {e}")
        return False


# --- Real agent generates skills ---
skill_writer = Agent(
    name="SkillWriter",
    instructions=(
        "You are a Python skill generator. Given a task description, write:\n"
        "1. A Python function that solves the task\n"
        "2. Assert-based test code that verifies it\n\n"
        "Output ONLY in this exact format (no markdown fences):\n"
        "# FUNCTION\n<function code>\n# TESTS\n<test assertions>"
    ),
)


async def generate_and_verify_skill(task: str, lib: SkillLibrary) -> SkillLibrary:
    """Use an LLM to generate a skill, verify it, and store if it passes."""
    # Check library first (reuse)
    match = lib.find(task)
    if match:
        print(f"  REUSE: Found existing skill '{match.name}'")
        return lib

    # Generate with LLM
    result = await Runner.run(skill_writer, f"Write a Python skill for: {task}")
    output = result.final_output

    # Parse function and tests
    if "# FUNCTION" in output and "# TESTS" in output:
        parts = output.split("# TESTS")
        func_code = parts[0].replace("# FUNCTION", "").strip()
        test_code = parts[1].strip()
    else:
        func_code = output.strip()
        test_code = ""

    print(f"  Generated: {func_code[:80]}...")

    # Verify
    if test_code and verify_skill(func_code, test_code):
        skill = Skill(
            name=task.lower().replace(" ", "_")[:30],
            description=task,
            code=func_code,
            test_code=test_code,
        )
        lib = lib.add(skill)
        print(f"  ✓ Verified and stored (library size: {len(lib.skills)})")
    else:
        print(f"  ✗ Verification failed or no tests — skill NOT stored")

    return lib


# Build a skill library with LLM-generated skills
print("=== Voyager: LLM-Generated Skill Library ===\n")
agent_library = SkillLibrary()

tasks = [
    "Write a function that checks if a string is a palindrome",
    "Write a function that flattens a nested list",
    "Check if a string is a palindrome",  # Should reuse!
]

for task in tasks:
    print(f"Task: {task}")
    agent_library = await generate_and_verify_skill(task, agent_library)
    print()

print(f"Final library size: {len(agent_library.skills)} skills")


### 2.6 GEPA Framework — Adaptive Re-Planning

**Goal → Environment → Plan → Action** — a continuous loop where the agent re-evaluates its plan after every action based on updated observations. Unlike static plans (Plan-Execute from Part 1), GEPA agents **re-plan after every step** to handle environments that change unpredictably.

This contrasts sharply with **Plan & Solve**, which commits to a full plan upfront and executes it sequentially. Plan & Solve works well when the task structure is predictable; GEPA trades that efficiency for adaptability when the environment shifts between steps.

**Use case:** Organizing a project workspace adaptively. The agent observes uncategorized files, asks an LLM to decide the best category for each file, executes the move, then re-observes before planning the next action. If new files appear mid-process, the agent adapts.

**When to use:** Dynamic environments where observations change after each action (file systems, live APIs, interactive debugging).
**When NOT to use:** Stable, predictable tasks where a static plan suffices — the re-planning overhead isn't worth it.

```
┌─────────────────────────────────────────────┐
│  1. GOAL — what am I trying to achieve?     │
│  2. ENVIRONMENT — what do I observe now?    │
│  3. PLAN — given goal + env, what's next?   │  ← LLM decides
│  4. ACTION — execute one step              │
│  └──→ loop back to step 2                   │
└─────────────────────────────────────────────┘
```

> **References:** [Plan & Solve Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/plan-and-solve.html) | [GEPA Framework](https://arxiv.org/abs/2305.13246)

In [ ]:
from agents import Agent, Runner
from dataclasses import dataclass, replace as dc_replace, field


@dataclass(frozen=True)
class FileItem:
    name: str
    category: str = "uncategorized"


@dataclass(frozen=True)
class GEPAState:
    """Immutable state for a GEPA agent organizing files."""
    goal: str
    files: tuple[FileItem, ...] = ()
    organized: dict[str, tuple[str, ...]] = field(default_factory=dict)
    actions_taken: tuple[str, ...] = ()
    done: bool = False


# --- LLM-based planner agent for the PLAN step ---
planner_agent = Agent(
    name="GEPAPlanner",
    instructions=(
        "You are a file organization planner. Given a list of uncategorized files and a goal, "
        "decide what category the next file should go into and explain your reasoning in one sentence. "
        "Respond in this exact format:\n"
        "CATEGORY: <category>\nREASON: <one sentence>"
    ),
)


async def gepa_loop(state: GEPAState, max_steps: int = 10) -> GEPAState:
    """Run the GEPA loop with LLM-based planning."""
    for step in range(1, max_steps + 1):
        # --- ENVIRONMENT: observe current state ---
        uncategorized = [f for f in state.files if f.category == "uncategorized"]
        if not uncategorized:
            state = dc_replace(state, done=True)
            print(f"\n✓ All files organized in {step - 1} steps")
            return state

        target = uncategorized[0]
        print(f"\nStep {step}: {len(uncategorized)} files remaining")
        print(f"  OBSERVE: Next file = '{target.name}'")

        # --- PLAN: ask LLM to decide category ---
        already = list(state.organized.keys()) if state.organized else ["(none yet)"]
        prompt = (
            f"Goal: {state.goal}\n"
            f"Existing categories: {', '.join(already)}\n"
            f"Remaining files: {', '.join(f.name for f in uncategorized)}\n"
            f"Decide the category for: {target.name}"
        )
        result = await Runner.run(planner_agent, prompt)
        output = result.final_output.strip()
        print(f"  PLAN: {output}")

        # Parse category from LLM output
        category = "other"
        for line in output.split("\n"):
            if line.strip().upper().startswith("CATEGORY:"):
                category = line.split(":", 1)[1].strip().lower()
                break

        # --- ACTION: categorize the file ---
        updated_file = FileItem(name=target.name, category=category)
        new_files = tuple(updated_file if f.name == target.name else f for f in state.files)
        existing = state.organized.get(category, ())
        new_organized = {**state.organized, category: (*existing, target.name)}

        state = dc_replace(
            state,
            files=new_files,
            organized=new_organized,
            actions_taken=(*state.actions_taken, f"Moved '{target.name}' → {category}"),
        )
        print(f"  ACTION: Moved '{target.name}' → {category}")

    return state


# --- Run the GEPA loop ---
initial_files = (
    FileItem("app.py"), FileItem("README.md"), FileItem("logo.png"),
    FileItem("config.yaml"), FileItem("setup.cfg"), FileItem("Dockerfile"),
    FileItem("test_app.py"), FileItem(".gitignore"), FileItem("CHANGELOG.md"),
)

initial_state = GEPAState(
    goal="Organize project files by type into logical categories",
    files=initial_files,
)

print("=== GEPA: Adaptive File Organization ===")
final_state = await gepa_loop(initial_state)

print(f"\nFinal organization:")
for cat, files in sorted(final_state.organized.items()):
    print(f"  {cat}/: {', '.join(files)}")
print(f"Total actions: {len(final_state.actions_taken)}")


### 2.7 Language Agent Tree Search (LATS)

Instead of linear retries (try once, fail, try again), LATS explores a **tree of candidate solutions**. At each step it generates multiple variants, scores them using an LLM evaluator, and uses **UCT (Upper Confidence bound for Trees)** to balance exploration vs exploitation. The `c` parameter (typically √2 ≈ 1.41) controls this balance — higher values favor exploring unvisited branches, lower values favor exploiting high-scoring ones.

LATS is the **most expensive** pattern in the taxonomy. Each iteration expands multiple nodes, and each node requires an LLM call for generation plus another for evaluation. Reserve it for problems where solution quality justifies the computational investment.

**Use case:** Complex code generation where multiple approaches might work. Instead of committing to one approach and iterating (Reflexion), LATS explores several approaches in parallel and drills deeper into the most promising one.

**When to use:** Multi-step problems where early greedy choices frequently lead to dead ends, and where you need the best solution, not just a good one.
**When NOT to use:** Simple tasks, budget-constrained scenarios, or when the first reasonable answer suffices. Use Reflexion (depth-first learning) instead when you want to iterate on a single approach.

```
                    Root (task)
                   /     |     \         
              Sol_A    Sol_B    Sol_C     ← generate N candidates
             score=0.6 score=0.8 score=0.3
                        |
                   /    |    \         
               B1      B2     B3        ← expand most promising
             s=0.7   s=0.9  s=0.5
```

> **References:** [LATS Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/lats.html) | [LATS Paper](https://arxiv.org/abs/2310.04406) | [LATS GitHub](https://github.com/andyz245/LanguageAgentTreeSearch)

In [ ]:
import math
from agents import Agent, Runner
from dataclasses import dataclass, replace as dc_replace


@dataclass(frozen=True)
class TreeNode:
    """Immutable LATS tree node."""
    solution: str
    score: float = 0.0
    visits: int = 0
    children: tuple["TreeNode", ...] = ()
    depth: int = 0

    def with_children(self, children: tuple["TreeNode", ...]) -> "TreeNode":
        return dc_replace(self, children=children)


def uct_score(node: TreeNode, parent_visits: int, c: float = 1.41) -> float:
    """Upper Confidence bound for Trees — balances exploration vs exploitation.

    c = √2 ≈ 1.41 is the standard exploration constant. Increase c to explore
    more unvisited branches; decrease to exploit high-scoring ones.
    """
    if node.visits == 0:
        return float("inf")
    exploitation = node.score
    exploration = c * math.sqrt(math.log(parent_visits) / node.visits)
    return exploitation + exploration


def select_best_leaf(node: TreeNode) -> TreeNode:
    """Select the leaf with highest UCT score (recursive)."""
    if not node.children:
        return node
    best_child = max(node.children, key=lambda c: uct_score(c, node.visits))
    return select_best_leaf(best_child)


# --- Real agents for generation and evaluation ---
generator_agent = Agent(
    name="SolutionGenerator",
    instructions=(
        "You are a code variant generator. Given a task and a parent solution, "
        "propose 3 DIFFERENT improved variants. Each variant should try a distinct approach. "
        "Output ONLY in this format (no markdown fences):\n"
        "VARIANT 1: <one-line description>\n"
        "VARIANT 2: <one-line description>\n"
        "VARIANT 3: <one-line description>"
    ),
)

evaluator_agent = Agent(
    name="SolutionEvaluator",
    instructions=(
        "You are a code solution evaluator. Given a task and a proposed solution approach, "
        "rate its quality on a scale of 0.0 to 1.0 based on correctness, efficiency, and elegance. "
        "Output ONLY a single decimal number like: 0.75"
    ),
)


async def agent_generate_variants(parent: TreeNode, task: str) -> tuple[TreeNode, ...]:
    """Use an LLM to generate solution variants."""
    result = await Runner.run(
        generator_agent,
        f"Task: {task}\nParent solution: {parent.solution}\nGenerate 3 improved variants."
    )
    lines = [l.strip() for l in result.final_output.strip().split("\n") if l.strip()]
    children = []
    for line in lines[:3]:
        desc = line.split(":", 1)[-1].strip() if ":" in line else line
        # Evaluate each variant with the evaluator agent
        eval_result = await Runner.run(
            evaluator_agent,
            f"Task: {task}\nProposed solution: {desc}\nRate quality 0.0-1.0."
        )
        try:
            score = float(eval_result.final_output.strip())
            score = max(0.0, min(1.0, score))
        except ValueError:
            score = 0.5
        children.append(TreeNode(
            solution=desc, score=score, visits=1, depth=parent.depth + 1
        ))
    return tuple(children)


def count_nodes(node: TreeNode) -> int:
    return 1 + sum(count_nodes(c) for c in node.children)


def find_best_solution(node: TreeNode) -> TreeNode:
    """Find highest-scoring node in the entire tree."""
    best = node
    for child in node.children:
        candidate = find_best_solution(child)
        if candidate.score > best.score:
            best = candidate
    return best


async def lats_search(task: str, max_iterations: int = 2) -> TreeNode:
    """Run Language Agent Tree Search with real agents."""
    root = TreeNode(solution="naive implementation", score=0.4, visits=1)

    for iteration in range(max_iterations):
        leaf = select_best_leaf(root)
        children = await agent_generate_variants(leaf, task)

        # Update tree (immutable rebuild)
        if leaf is root:
            root = root.with_children(children)
        else:
            new_root_children = []
            for child in root.children:
                if child is leaf:
                    sub_children = await agent_generate_variants(child, task)
                    new_root_children.append(child.with_children(sub_children))
                else:
                    new_root_children.append(child)
            root = root.with_children(tuple(new_root_children))

        print(f"Iteration {iteration + 1}: tree has {count_nodes(root)} nodes")

    return root


# --- Run LATS ---
task = "Write a function to merge two sorted arrays efficiently"
result_tree = await lats_search(task, max_iterations=2)
best = find_best_solution(result_tree)

print(f"\nTask: {task}")
print(f"Best solution: '{best.solution}'")
print(f"Score: {best.score:.3f} (depth={best.depth})")
print(f"Total nodes explored: {count_nodes(result_tree)}")


def print_tree(node: TreeNode, indent: int = 0):
    marker = " ★" if node is best else ""
    print(f"{'  ' * indent}[{node.score:.2f}] {node.solution[:60]}{marker}")
    for child in node.children:
        print_tree(child, indent + 1)


print(f"\n--- Tree structure ---")
print_tree(result_tree)



---

## Part 3: Advanced Agent Strategies — Production Patterns

Parts 1–2 focused on orchestration and learning. Part 3 shifts to **production reliability**: error recovery, decision-making under constraints, hierarchical decomposition, and meta-reasoning.

| Pattern | Key idea |
|---------|----------|
| Error Recovery (§3.1) | Layered retry → fallback → degrade |
| Utility-Based (§3.2) | Score plans by quality/cost/speed |
| HTN (§3.3) | Recursive task decomposition |
| Causal Discovery (§3.4) | LLM proposes, statistics validates |
| REWOO (§3.5) | Plan all tool calls upfront, execute in batch |
| Self-Discovery (§3.6) | Agent selects its own reasoning strategy |

### 3.1 Composable Error Recovery Strategies

Real production systems need **layered recovery**: retry the primary agent N times, then fall back to a simpler agent, then degrade gracefully with a default response. Each strategy is a composable building block that you chain together.

This complements Reflexion (1.4). Reflexion recovers through **learning** — analyzing what went wrong and revising the approach. Error recovery strategies recover through **mechanical resilience** — retrying, switching, degrading. In production, you compose both: Reflexion for quality improvement, error recovery for operational reliability.

**Use case:** A code review service where the detailed analyzer might hit rate limits. Chain: Retry detailed analyzer 3× → Fall back to quick analyzer → Degrade to "service unavailable."

| Strategy | Behavior | When to Use |
|----------|----------|-------------|
| `Retry(n)` | Re-run same agent N times | Transient failures (rate limits, timeouts) |
| `Fallback(alt)` | Switch to alternative agent | Primary agent can't handle this task |
| `Degrade(msg)` | Return simplified output | All agents failed, need _something_ |

> **References:** [Retry Pattern (Azure)](https://learn.microsoft.com/en-us/azure/architecture/patterns/retry) | [Circuit Breaker Pattern](https://learn.microsoft.com/en-us/azure/architecture/patterns/circuit-breaker)

In [ ]:
from agents import Agent, Runner
from dataclasses import dataclass, replace as dc_replace


@dataclass(frozen=True)
class Result:
    """Immutable result of an agent invocation."""
    success: bool
    output: str
    strategy_used: str
    attempts: int = 1


@dataclass(frozen=True)
class Retry:
    max_attempts: int = 3

    async def execute(self, agent: Agent, prompt: str, label: str = "primary") -> Result:
        for attempt in range(1, self.max_attempts + 1):
            try:
                result = await Runner.run(agent, prompt)
                return Result(success=True, output=result.final_output,
                              strategy_used=f"retry({label})", attempts=attempt)
            except Exception as e:
                print(f"  Attempt {attempt}/{self.max_attempts} failed: {e}")
                if attempt == self.max_attempts:
                    return Result(success=False, output=str(e),
                                  strategy_used=f"retry({label})-exhausted",
                                  attempts=attempt)
        return Result(success=False, output="unreachable",
                      strategy_used="retry", attempts=self.max_attempts)


@dataclass(frozen=True)
class Fallback:
    label: str = "fallback"

    async def execute(self, agent: Agent, prompt: str, primary_result: Result) -> Result:
        if primary_result.success:
            return primary_result
        try:
            result = await Runner.run(agent, prompt)
            return Result(success=True, output=result.final_output,
                          strategy_used=self.label, attempts=1)
        except Exception as e:
            return Result(success=False, output=str(e),
                          strategy_used=f"{self.label}-failed", attempts=1)


@dataclass(frozen=True)
class Degrade:
    degraded_output: str = "[Degraded] Unable to process. Returning default."

    def execute(self, prev_result: Result) -> Result:
        if prev_result.success:
            return prev_result
        return Result(success=True, output=self.degraded_output,
                      strategy_used="degrade", attempts=0)


# --- Real agents with different capability levels ---
primary_agent = Agent(
    name="DetailedAnalyzer",
    instructions=(
        "Provide a detailed, thorough code review covering security, performance, "
        "and maintainability. Be specific with line-level recommendations."
    ),
)

fallback_agent = Agent(
    name="QuickAnalyzer",
    instructions=(
        "Provide a brief, high-level code review in 3-5 bullet points. "
        "Focus on the most critical issues only."
    ),
)


async def compose_recovery(
    prompt: str,
    primary: Agent,
    fallback: Agent,
    degraded_msg: str = "[Degraded] Service unavailable",
    max_retries: int = 2,
) -> Result:
    """Compose: Retry primary → Fallback to simpler agent → Degrade."""
    result = await Retry(max_retries).execute(primary, prompt, label="primary")
    result = await Fallback(label="fallback-agent").execute(fallback, prompt, result)
    result = Degrade(degraded_msg).execute(result)
    return result


# --- Demo with a real code review task ---
code_to_review = '''
def get_user(user_id):
    query = f"SELECT * FROM users WHERE id = {user_id}"
    result = db.execute(query)
    data = []
    for row in result:
        data.append(row)
    return data
'''

print("=== Composable Error Recovery with Real Agents ===\n")
result = await compose_recovery(
    prompt=f"Review this code:\n```python\n{code_to_review}\n```",
    primary=primary_agent,
    fallback=fallback_agent,
)
print(f"Strategy used: {result.strategy_used}")
print(f"Attempts: {result.attempts}")
# print(f"\nOutput:\n{result.output[:500]}")

### 3.2 Utility-Based Decision Making

Most agents blindly execute whatever approach comes to mind. **Utility-based agents** evaluate multiple candidate plans by scoring quality, cost, and speed tradeoffs, then execute only the plan with the highest utility score. This is the **meta-pattern** for choosing between strategies — the agent-level equivalent of the pattern comparison matrix at the top of this section.

In the agent-patterns taxonomy, this maps to how you'd programmatically select between REWOO (cost-efficient) vs LATS (quality-maximizing) vs Reflection (balanced) based on task requirements and resource constraints.

**Use case:** Choosing between a fast-cheap analysis (gpt-4o-mini, 2 sentences) vs a thorough analysis (gpt-4o, detailed report). Under cost pressure, pick the cheap option; when quality matters, pick the thorough one.

**When to use:** Tasks with competing objectives (quality vs cost vs speed) where the right approach depends on context. Especially valuable when the same system handles both low-stakes and high-stakes requests.

```
              ┌─ Plan A → utility(quality=0.9, cost=500, time=30s) = 0.72
Task ────────├─ Plan B → utility(quality=0.7, cost=100, time=10s) = 0.81 ✓
              └─ Plan C → utility(quality=0.6, cost=50,  time=5s)  = 0.74
```

> **References:** [Choosing a Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/choosing-a-pattern.html) | [Utility-Based Agents (Russell & Norvig)](https://aima.cs.berkeley.edu/)

In [ ]:
from agents import Agent, Runner
from dataclasses import dataclass


# Utility-based selection: the meta-pattern for choosing between strategies.
# This is how you'd programmatically pick REWOO (cheap) vs LATS (thorough)
# vs Reflection (balanced) based on task requirements.

@dataclass(frozen=True)
class UtilityWeights:
    """Weights for the utility function — must sum to 1.0."""
    quality: float = 0.5
    cost: float = 0.3
    speed: float = 0.2


@dataclass(frozen=True)
class Plan:
    name: str
    estimated_quality: float   # 0.0 - 1.0
    estimated_cost: int        # tokens
    estimated_time: float      # seconds
    model: str = "gpt-4o-mini"


def compute_utility(plan: Plan, weights: UtilityWeights,
                    max_cost: int = 1000, max_time: float = 120.0) -> float:
    """Compute normalized utility score for a plan."""
    quality_score = plan.estimated_quality
    cost_score = 1.0 - (plan.estimated_cost / max_cost)
    speed_score = 1.0 - (plan.estimated_time / max_time)
    return (
        weights.quality * quality_score
        + weights.cost * max(0, cost_score)
        + weights.speed * max(0, speed_score)
    )


# --- Define candidate plans ---
plans = (
    Plan("Quick & Cheap", estimated_quality=0.6, estimated_cost=80, estimated_time=5,
         model="gpt-4o-mini"),
    Plan("Balanced", estimated_quality=0.8, estimated_cost=300, estimated_time=20,
         model="gpt-4o"),
    Plan("Maximum Quality", estimated_quality=0.95, estimated_cost=700, estimated_time=45,
         model="o1"),
)

# --- Score under different weight presets ---
presets = {
    "Cost-Optimized":    UtilityWeights(quality=0.2, cost=0.6, speed=0.2),
    "Quality-Optimized": UtilityWeights(quality=0.7, cost=0.1, speed=0.2),
    "Speed-Optimized":   UtilityWeights(quality=0.2, cost=0.2, speed=0.6),
    "Balanced":          UtilityWeights(quality=0.4, cost=0.3, speed=0.3),
}

print(f"{'Preset':<22} {'Selected Plan':<25} {'Utility':>8}")
print("-" * 58)
for preset_name, weights in presets.items():
    scored = [(p, compute_utility(p, weights)) for p in plans]
    scored.sort(key=lambda x: x[1], reverse=True)
    best_plan, best_score = scored[0]
    print(f"{preset_name:<22} {best_plan.name:<25} {best_score:>8.3f}")

# --- Execute the winning plan with a real agent ---
print("\n=== Executing Best Plan Under 'Balanced' Preset ===")
balanced_weights = presets["Balanced"]
winner = max(plans, key=lambda p: compute_utility(p, balanced_weights))
print(f"Selected: {winner.name} (model={winner.model})\n")

# Create agents matching different quality/cost tradeoffs
agents_by_plan = {
    "Quick & Cheap": Agent(
        name="QuickAnalyzer", model="gpt-4o-mini",
        instructions="Give a brief, 2-sentence analysis. Prioritize speed over depth.",
    ),
    "Balanced": Agent(
        name="BalancedAnalyzer",
        instructions="Give a thorough but concise analysis in 1-2 paragraphs.",
    ),
    "Maximum Quality": Agent(
        name="ThoroughAnalyzer",
        instructions=(
            "Give a detailed, comprehensive analysis covering all angles. "
            "Include specific recommendations with code examples."
        ),
    ),
}

task = (
    "Analyze this Python function for potential issues: "
    "def get_user(id): return db.query(f'SELECT * FROM users WHERE id={id}')"
)

selected_agent = agents_by_plan[winner.name]
result = await Runner.run(selected_agent, task)
print(f"Agent: {selected_agent.name}")
print(f"Output ({len(result.final_output)} chars):\n{result.final_output[:500]}")



### 3.3 Hierarchical Task Networks (HTN)

HTN planning decomposes **compound tasks** into **primitive subtasks** recursively, forming a tree. Instead of hardcoding the decomposition rules, an LLM agent decides how to break down each compound task. This is the recursive, hierarchical version of **Plan & Solve** — where Plan & Solve creates a flat step list, HTN creates a tree of nested subtasks.

Compare with REWOO (1.12): HTN focuses on **task structure** (breaking down what needs to be done), while REWOO focuses on **tool orchestration** (planning which tools to call with which arguments). HTN answers "what are the subtasks?", REWOO answers "what are the tool calls?"

**Use case:** "Deploy a new feature" → create branch → implement → write tests → open PR → code review → merge. The LLM understands software engineering processes and produces contextually appropriate decompositions.

**When to use:** Well-structured domains where tasks have natural hierarchical decomposition (software engineering, project management, multi-step workflows).

```
Deploy Feature (compound)
├── Write Code (compound)
│   ├── Create branch (primitive)
│   ├── Implement logic (primitive)
│   └── Write tests (primitive)
├── Review & Merge (compound)
│   ├── Open PR (primitive)
│   ├── Code review (primitive)
│   └── Merge (primitive)
└── Update Docs (primitive)
```

> **References:** [Plan & Solve Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/plan-and-solve.html) | [Wikipedia — HTN Planning](https://en.wikipedia.org/wiki/Hierarchical_task_network)

In [ ]:
from agents import Agent, Runner
from dataclasses import dataclass, replace as dc_replace
from enum import Enum
import re


class TaskType(Enum):
    PRIMITIVE = "primitive"
    COMPOUND = "compound"


@dataclass(frozen=True)
class HTNTask:
    """Immutable HTN task node."""
    name: str
    task_type: TaskType
    subtasks: tuple["HTNTask", ...] = ()
    status: str = "pending"
    result: str = ""

    def with_status(self, status: str, result: str = "") -> "HTNTask":
        return dc_replace(self, status=status, result=result)


# --- LLM-based decomposition agent ---
decomposer = Agent(
    name="TaskDecomposer",
    instructions=(
        "You are a task decomposition expert. Given a compound task, break it into "
        "2-4 concrete subtasks that a developer would execute sequentially. "
        "Output ONLY a numbered list, one subtask per line. Example:\n"
        "1. Create feature branch from main\n"
        "2. Implement the login form component\n"
        "3. Write unit tests for the login form\n"
        "Do NOT include explanations — just the numbered list."
    ),
)


async def llm_decompose(task_name: str) -> tuple[HTNTask, ...]:
    """Ask an LLM to decompose a compound task into subtasks."""
    result = await Runner.run(
        decomposer,
        f"Decompose this software engineering task into concrete subtasks: '{task_name}'"
    )
    lines = result.final_output.strip().split("\n")
    subtasks = []
    for line in lines:
        cleaned = re.sub(r"^\d+\.\s*", "", line.strip())
        if cleaned:
            subtasks.append(HTNTask(cleaned, TaskType.PRIMITIVE))
    return tuple(subtasks)


async def execute_htn(task: HTNTask, depth: int = 0) -> HTNTask:
    """Execute an HTN plan depth-first with LLM decomposition."""
    indent = "  " * depth
    print(f"{indent}→ {task.name} ({task.task_type.value})")

    if task.task_type == TaskType.PRIMITIVE:
        completed = task.with_status("done", result=f"Completed: {task.name}")
        print(f"{indent}  ✓ {completed.result}")
        return completed

    # Decompose compound task using LLM
    subtasks = await llm_decompose(task.name)
    task = dc_replace(task, subtasks=subtasks)

    # Execute subtasks sequentially
    completed_subtasks = []
    for subtask in task.subtasks:
        completed = await execute_htn(subtask, depth + 1)
        completed_subtasks.append(completed)

    return dc_replace(
        task,
        subtasks=tuple(completed_subtasks),
        status="done",
        result=f"All {len(completed_subtasks)} subtasks completed",
    )


# --- Run HTN with LLM decomposition ---
compound_tasks = [
    "Deploy a new user authentication feature",
    "Set up CI/CD pipeline for a Python project",
]

for task_name in compound_tasks:
    print(f"\n{'='*50}")
    root = HTNTask(task_name, TaskType.COMPOUND)
    completed = await execute_htn(root)

    def count_primitives(t: HTNTask) -> int:
        if t.task_type == TaskType.PRIMITIVE:
            return 1
        return sum(count_primitives(s) for s in t.subtasks)

    print(f"\nPrimitive tasks: {count_primitives(completed)}")
    print(f"Status: {completed.status}")


### 3.4 LLM-Assisted Causal Discovery

Most agents treat the world as correlations — "X went up when Y went up." But correlation-based reasoning fails when confounders exist: two variables may move together only because a hidden third variable drives both. **Causal discovery** combines LLM domain knowledge (to propose plausible cause→effect relationships) with statistical validation (to confirm them against real data).

This matters for agent decision-making because agents trained on correlated data will make spurious decisions. If code reviews and bug count are both correlated with team size, an agent might wrongly conclude "more code reviews = more bugs" instead of recognizing that team size drives both.

**Use case:** "Does more code review actually reduce bugs, or is it just correlated via team size?" An LLM proposes the causal graph from variable names, then partial correlations validate each edge.

```
LLM: "team_size → code_reviews"       (domain knowledge)
     "code_reviews → bug_count"       
     "team_size → bug_count"          

Data: team_size → code_reviews   ✓ p < 0.05
      code_reviews → bug_count   ✓ p < 0.05
      team_size → bug_count      ✓ p < 0.05
```

> **References:** [DoWhy Library](https://www.pywhy.org/dowhy/) | [Causal Inference in Statistics (Pearl)](http://bayes.cs.ucla.edu/PRIMER/)

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats
import re
from agents import Agent, Runner

# --- Step 1: Generate synthetic software engineering data ---
np.random.seed(42)
n = 200

team_size = np.random.randint(3, 20, n)
code_reviews = (team_size * 0.8 + np.random.normal(0, 2, n)).clip(1).astype(int)
test_coverage = (code_reviews * 3.5 + np.random.normal(60, 10, n)).clip(10, 100)
bug_count = (50 - code_reviews * 1.5 - test_coverage * 0.2 + team_size * 0.3
             + np.random.normal(0, 5, n)).clip(0).astype(int)

df = pd.DataFrame({
    "team_size": team_size,
    "code_reviews": code_reviews,
    "test_coverage": test_coverage.round(1),
    "bug_count": bug_count,
})
print(f"Dataset: {len(df)} rows, columns: {list(df.columns)}")
print(df.describe().round(1))

# --- Step 2: Ask LLM to propose causal relationships ---
causal_discovery_agent = Agent(
    name="CausalDiscoverer",
    instructions=(
        "You are a causal inference expert. Given variable names from a software "
        "engineering dataset, propose directed causal relationships.\n"
        "Output ONLY lines in the format: cause -> effect\n"
        "Consider domain knowledge about software engineering processes.\n"
        "Do NOT include any other text, just the arrows."
    ),
)

variables = list(df.columns)
result = await Runner.run(
    causal_discovery_agent,
    f"Variables: {variables}\nPropose all plausible causal relationships:",
)

print("\n=== LLM-Proposed Causal Graph ===")
print(result.final_output)

# --- Step 3: Parse the LLM's proposed edges ---
proposed_edges = []
for line in result.final_output.strip().split("\n"):
    match = re.match(r"(\w+)\s*->\s*(\w+)", line.strip())
    if match:
        proposed_edges.append((match.group(1), match.group(2)))

print(f"\nParsed {len(proposed_edges)} edges")

# --- Step 4: Validate each edge against data using partial correlation ---
print(f"\n=== Statistical Validation (p < 0.05) ===")
print(f"{'Edge':<40} {'Correlation':>11} {'p-value':>10} {'Valid?':>7}")
print("-" * 70)

validated_edges = []
for cause, effect in proposed_edges:
    if cause in df.columns and effect in df.columns:
        corr, p_val = stats.pearsonr(df[cause], df[effect])
        is_valid = p_val < 0.05
        status = "✓" if is_valid else "✗"
        print(f"{cause:>18} → {effect:<18} {corr:>+11.3f} {p_val:>10.4f} {status:>7}")
        if is_valid:
            validated_edges.append((cause, effect))
    else:
        print(f"{cause:>18} → {effect:<18} {'N/A':>11} {'N/A':>10} {'SKIP':>7}")

print(f"\nValidated: {len(validated_edges)}/{len(proposed_edges)} edges")
print(f"\nValidated DOT graph:")
print("digraph {")
for c, e in validated_edges:
    print(f"    {c} -> {e};")
print("}")


### 3.5 REWOO — Plan Once, Execute in Parallel

**Reasoning Without Observation (REWOO)** separates planning from execution entirely. A **planner** agent generates ALL tool calls upfront with placeholder variables (`#E1`, `#E2`), tools execute respecting dependencies, and a **solver** agent integrates all results into a final answer. This makes exactly two LLM calls regardless of how many tools are involved — dramatically cheaper than ReAct's iterative reasoning-action loop.

The key insight: when you know which tools you need and in what order, there's no reason to interleave LLM reasoning between each tool call. Plan the full sequence once, execute it, then synthesize.

**Use case:** Multi-step research — search for a topic, fetch related articles, extract key data, then synthesize findings. Instead of 8 LLM calls (ReAct: think→search→think→fetch→think→extract→think→synthesize), REWOO uses 2 (plan→synthesize) plus the tool calls themselves.

**When to use:** Tool-heavy workflows with predictable dependencies, cost-sensitive applications, API orchestration.
**When NOT to use:** Exploratory tasks where the next tool depends on the previous result's content, or when you can't predict the tool sequence upfront.

```
Planner LLM:                          Execution:
  #E1 = search("topic")                 #E1 → "search results..."
  #E2 = fetch_url(#E1)                  #E2 → "page content..."  (waits for #E1)
  #E3 = extract_data(#E2)               #E3 → "extracted data..."

Solver LLM:
  Task + #E1 + #E2 + #E3 → Final Answer
```

> **References:** [REWOO Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/rewoo.html) | [REWOO Paper](https://arxiv.org/abs/2305.18323)


In [ ]:
import re
from agents import Agent, Runner, function_tool
from dataclasses import dataclass, replace as dc_replace


@dataclass(frozen=True)
class ToolCall:
    """A planned tool call with placeholder dependencies."""
    var_id: str         # e.g., "#E1"
    tool_name: str      # e.g., "search"
    raw_args: str       # e.g., "machine learning trends"
    dependencies: tuple[str, ...] = ()  # e.g., ("#E1",)


# --- Simulated tools ---
def _search(query: str) -> str:
    """Search for information on a topic."""
    return f"[Search results for '{query}': Recent advances include transformer architectures, RLHF, and multi-modal models.]"


def _summarize_text(text: str) -> str:
    """Summarize a piece of text into key points."""
    return f"[Summary: {text[:80]}... Key themes: architecture, training, applications.]"


def _compare(item_a: str, item_b: str) -> str:
    """Compare two items and highlight differences."""
    return f"[Comparison: '{item_a}' vs '{item_b}' — differ in scale, cost, and accuracy tradeoffs.]"


# Wrap for agent use
search = function_tool(_search)
summarize_text = function_tool(_summarize_text)
compare = function_tool(_compare)

# Raw functions for direct REWOO execution (bypasses agent runtime)
TOOLS = {"search": _search, "summarize_text": _summarize_text, "compare": _compare}


# --- Planner agent: generates the full tool-call plan ---
planner = Agent(
    name="REWOOPlanner",
    instructions=(
        "You are a task planner. Given a task, create a plan of tool calls "
        "using placeholder variables. Available tools: search, summarize_text, compare.\n\n"
        "Output ONLY in this exact format (no explanation):\n"
        "#E1 = search(\"query here\")\n"
        "#E2 = summarize_text(#E1)\n"
        "#E3 = compare(\"item a\", \"item b\")\n"
        "...\n\n"
        "Use #E1, #E2, etc. as placeholders for previous results."
    ),
)


# --- Solver agent: integrates tool results into final answer ---
solver = Agent(
    name="REWOOSolver",
    instructions=(
        "You are a research synthesizer. Given a task and a set of tool results, "
        "produce a clear, comprehensive answer that integrates all the evidence. "
        "Be concise but thorough."
    ),
)


def parse_plan(plan_text: str) -> tuple[ToolCall, ...]:
    """Parse planner output into structured tool calls."""
    calls = []
    for line in plan_text.strip().split("\n"):
        match = re.match(r"(#E\d+)\s*=\s*(\w+)\((.+)\)", line.strip())
        if match:
            var_id = match.group(1)
            tool_name = match.group(2)
            raw_args = match.group(3)
            # Find dependencies: any #E references in the args
            deps = tuple(re.findall(r"#E\d+", raw_args))
            calls.append(ToolCall(var_id=var_id, tool_name=tool_name,
                                  raw_args=raw_args, dependencies=deps))
    return tuple(calls)


def resolve_args(raw_args: str, results: dict[str, str]) -> list[str]:
    """Replace placeholder variables with actual results."""
    resolved = raw_args
    for var_id, result in results.items():
        resolved = resolved.replace(var_id, result)
    # Split on commas, strip quotes
    parts = [p.strip().strip('"').strip("'") for p in resolved.split('",')]
    return parts


async def rewoo_execute(task: str) -> str:
    """Run the full REWOO pipeline: Plan → Execute → Integrate."""

    # Phase 1: Plan — single LLM call
    print("=== Phase 1: PLAN ===")
    plan_result = await Runner.run(planner, f"Create a tool-call plan for: {task}")
    print(plan_result.final_output)
    plan = parse_plan(plan_result.final_output)
    print(f"\nParsed {len(plan)} tool calls")

    # Phase 2: Execute — no LLM calls, just tools
    print("\n=== Phase 2: EXECUTE ===")
    results: dict[str, str] = {}
    for call in plan:
        # Resolve placeholders from previous results
        args = resolve_args(call.raw_args, results)
        tool_fn = TOOLS.get(call.tool_name)
        if tool_fn:
            # Call the raw function directly (bypasses agent runtime)
            if len(args) >= 2:
                result = tool_fn(args[0], args[1])
            else:
                result = tool_fn(args[0])
            results[call.var_id] = result
            print(f"  {call.var_id} = {call.tool_name}(...) → {result[:80]}...")
        else:
            results[call.var_id] = f"[Unknown tool: {call.tool_name}]"

    # Phase 3: Integrate — single LLM call
    print("\n=== Phase 3: INTEGRATE ===")
    evidence = "\n".join(f"{var}: {val}" for var, val in results.items())
    solver_prompt = f"Task: {task}\n\nTool Results:\n{evidence}\n\nSynthesize a final answer."
    final = await Runner.run(solver, solver_prompt)
    return final.final_output


# --- Run REWOO ---
answer = await rewoo_execute(
    "Compare the current state of transformer models vs diffusion models for AI"
)
print(f"\n=== FINAL ANSWER ===\n{answer[:500]}")

### 3.6 Self-Discovery — Adaptive Reasoning Module Selection

Most patterns commit to a single reasoning strategy upfront. **Self-Discovery** takes a meta-reasoning approach: the agent first **discovers** which reasoning strategies are appropriate for the task, **adapts** them to the specific problem, then **executes** using the tailored approach. It's the only pattern that dynamically selects its own methodology.

This is distinct from utility-based selection (1.9), which scores predefined plans. Self-Discovery selects from reasoning *methods* (analytical, creative, step-by-step, analogical) rather than execution *plans* (fast/cheap vs slow/thorough).

**Use case:** A complex design question like "How should we architect a real-time collaborative editor?" — the agent might select analytical reasoning (for system decomposition), analogical reasoning (drawing parallels to Google Docs), and step-by-step reasoning (for implementation planning), then combine insights from all three.

**When to use:** Complex multi-faceted problems where you don't know which reasoning approach will work best. Problems that benefit from perspective diversity.
**When NOT to use:** Simple well-defined tasks, time-critical scenarios, or tasks where the reasoning approach is obvious (e.g., use step-by-step for math, use analytical for debugging).

```
                     ┌─ Analytical reasoning ──┐
Task → Discoverer → ─┤─ Analogical reasoning ──├─ Adapter → Executor → Answer
                     └─ Step-by-step reasoning ─┘
```

> **References:** [Self-Discovery Pattern](https://agent-patterns.readthedocs.io/en/latest/patterns/self-discovery.html) | [Self-Discovery Paper](https://arxiv.org/abs/2402.03620)


In [ ]:
import re
from agents import Agent, Runner, function_tool
from dataclasses import dataclass, replace as dc_replace


@dataclass(frozen=True)
class ToolCall:
    """A planned tool call with placeholder dependencies."""
    var_id: str         # e.g., "#E1"
    tool_name: str      # e.g., "search"
    raw_args: str       # e.g., "machine learning trends"
    dependencies: tuple[str, ...] = ()  # e.g., ("#E1",)


# --- Simulated tools ---
def _search(query: str) -> str:
    """Search for information on a topic."""
    return f"[Search results for '{query}': Recent advances include transformer architectures, RLHF, and multi-modal models.]"


def _summarize_text(text: str) -> str:
    """Summarize a piece of text into key points."""
    return f"[Summary: {text[:80]}... Key themes: architecture, training, applications.]"


def _compare(item_a: str, item_b: str) -> str:
    """Compare two items and highlight differences."""
    return f"[Comparison: '{item_a}' vs '{item_b}' — differ in scale, cost, and accuracy tradeoffs.]"


# Wrap for agent use
search = function_tool(_search)
summarize_text = function_tool(_summarize_text)
compare = function_tool(_compare)

# Raw functions for direct REWOO execution (bypasses agent runtime)
TOOLS = {"search": _search, "summarize_text": _summarize_text, "compare": _compare}


# --- Planner agent: generates the full tool-call plan ---
planner = Agent(
    name="REWOOPlanner",
    instructions=(
        "You are a task planner. Given a task, create a plan of tool calls "
        "using placeholder variables. Available tools: search, summarize_text, compare.\n\n"
        "Output ONLY in this exact format (no explanation):\n"
        "#E1 = search(\"query here\")\n"
        "#E2 = summarize_text(#E1)\n"
        "#E3 = compare(\"item a\", \"item b\")\n"
        "...\n\n"
        "Use #E1, #E2, etc. as placeholders for previous results."
    ),
)


# --- Solver agent: integrates tool results into final answer ---
solver = Agent(
    name="REWOOSolver",
    instructions=(
        "You are a research synthesizer. Given a task and a set of tool results, "
        "produce a clear, comprehensive answer that integrates all the evidence. "
        "Be concise but thorough."
    ),
)


def parse_plan(plan_text: str) -> tuple[ToolCall, ...]:
    """Parse planner output into structured tool calls."""
    calls = []
    for line in plan_text.strip().split("\n"):
        match = re.match(r"(#E\d+)\s*=\s*(\w+)\((.+)\)", line.strip())
        if match:
            var_id = match.group(1)
            tool_name = match.group(2)
            raw_args = match.group(3)
            # Find dependencies: any #E references in the args
            deps = tuple(re.findall(r"#E\d+", raw_args))
            calls.append(ToolCall(var_id=var_id, tool_name=tool_name,
                                  raw_args=raw_args, dependencies=deps))
    return tuple(calls)


def resolve_args(raw_args: str, results: dict[str, str]) -> list[str]:
    """Replace placeholder variables with actual results, then extract args."""
    resolved = raw_args
    for var_id, result in results.items():
        resolved = resolved.replace(var_id, result)
    # Parse quoted strings and bare #E references as individual args
    parts = re.findall(r'"([^"]*?)"|\'([^\']*?)\'|(#E\d+)', resolved)
    args = [g0 or g1 or g2 for g0, g1, g2 in parts]
    # Fallback: if no quoted/placeholder args found, treat entire string as one arg
    if not args:
        args = [resolved.strip().strip('"').strip("'")]
    return args


async def rewoo_execute(task: str) -> str:
    """Run the full REWOO pipeline: Plan → Execute → Integrate."""

    # Phase 1: Plan — single LLM call
    print("=== Phase 1: PLAN ===")
    plan_result = await Runner.run(planner, f"Create a tool-call plan for: {task}")
    print(plan_result.final_output)
    plan = parse_plan(plan_result.final_output)
    print(f"\nParsed {len(plan)} tool calls")

    # Phase 2: Execute — no LLM calls, just tools
    print("\n=== Phase 2: EXECUTE ===")
    results: dict[str, str] = {}
    for call in plan:
        # Resolve placeholders from previous results
        args = resolve_args(call.raw_args, results)
        tool_fn = TOOLS.get(call.tool_name)
        if tool_fn:
            # Call the raw function directly (bypasses agent runtime)
            result = tool_fn(*args)
            results[call.var_id] = result
            print(f"  {call.var_id} = {call.tool_name}(...) → {result[:80]}...")
        else:
            results[call.var_id] = f"[Unknown tool: {call.tool_name}]"

    # Phase 3: Integrate — single LLM call
    print("\n=== Phase 3: INTEGRATE ===")
    evidence = "\n".join(f"{var}: {val}" for var, val in results.items())
    solver_prompt = f"Task: {task}\n\nTool Results:\n{evidence}\n\nSynthesize a final answer."
    final = await Runner.run(solver, solver_prompt)
    return final.final_output


# --- Run REWOO ---
answer = await rewoo_execute(
    "Compare the current state of transformer models vs diffusion models for AI"
)
print(f"\n=== FINAL ANSWER ===\n{answer[:500]}")

---

## Closing

This tutorial covered the full stack — from agent fundamentals (tools, memory, guardrails) through advanced multi-agent strategies.

### Three ways agents fail

| Failure | Cause | Fix |
|---------|-------|-----|
| Wrong plan | LLM picks the wrong direction | Better instructions; add a critic loop |
| Wrong tool call | Vague description → wrong arguments | Write precise, human-readable docstrings |
| Tool error | Tool returns an error or bad output | Return descriptive error strings from tools so the model can retry |

### References

| Resource | Link |
|----------|------|
| OpenAI Agents SDK | [openai.github.io/openai-agents-python](https://openai.github.io/openai-agents-python/) |
| SDK examples | [github.com/openai/openai-agents-python/tree/main/examples](https://github.com/openai/openai-agents-python/tree/main/examples) |
| LLM Powered Autonomous Agents (Weng, 2023) | [lilianweng.github.io](https://lilianweng.github.io/posts/2023-06-23-agent/) |
| ReAct (Yao et al., 2022) | [arxiv.org/abs/2210.03629](https://arxiv.org/abs/2210.03629) |
| Reflexion (Shinn et al., 2023) | [arxiv.org/abs/2303.11366](https://arxiv.org/abs/2303.11366) |



In [ ]:
# Cleanup temporary workspace
import shutil, os
if os.path.exists(WORKSPACE):
    shutil.rmtree(WORKSPACE)
    print(f"Cleaned up: {WORKSPACE}")
else:
    print("Workspace already cleaned up")